# 1008 26일차

## 1. 다중 입력 모델

입력을 여러 개 받아 입력마다 층을 따로 쌓은 뒤, 그 출력을 합쳐 다음 층으로 넘기는 함수형 모델 (`keras69_ensemble1`·`2`). 수업에서는 앙상블이라고 부름

`Sequential`은 입력을 하나만 받음. 열 구성이 다른 데이터를 입력마다 다른 층에 넣으려면 함수형 모델(day11 §4)로 만듦

1. `Input` 여러 개 : 데이터마다 `Input(shape=(열 개수,))`를 만들고 그 뒤에 층을 이음
2. `Concatenate` : 입력마다 쌓은 층의 마지막 출력을 이어 붙임. 출력 노드 수는 이어 붙인 출력의 노드 수를 더한 값
3. `Model(inputs=[input1, input2], outputs=...)` : 입력을 리스트로 지정

```python
merge1 = Concatenate(name='mg1')([output1, output2])   # 층 클래스 : 층을 만든 뒤 리스트를 넣어 호출
merge1 = concatenate([output1, output2], name='mg1')   # 함수 : 리스트를 바로 넣음
```

- `fit`·`evaluate`·`predict`의 x도 `inputs` 순서대로 `[x1, x2]` 리스트로 넣음
- 리스트로 묶지 않고 `predict(x1, x2)`로 넣으면 `x2`가 두 번째 인자 `batch_size` 자리에 들어감
- 입력끼리 행 수가 같아야 함. 같은 순서의 행끼리 한 샘플로 묶임
- `train_test_split(x1, x2, y)`처럼 배열을 여러 개 넣으면 모두 같은 행 번호로 나눔. `x1_train, x1_test, x2_train, x2_test, y_train, y_test` 순서로 돌려줌

#### cf) 앙상블

여러 모델이 같은 문제를 각자 예측하고, 그 예측값을 평균이나 투표로 합치는 방법

- 다중 입력 모델에서 입력마다 쌓은 층의 출력은 다음 층에 넘기는 중간값이라, 그 출력만으로는 예측값이 나오지 않음. 전체를 `fit` 한 번에 loss 하나로 같이 학습함
- 앙상블은 모델마다 예측이 달라야 평균 낼 때 오차 일부가 상쇄됨. 초기 가중치·학습 데이터·모델 종류를 바꿔 차이를 만듦
- Keras 가이드의 앙상블(ensembling) 예시는 같은 입력을 모델 여러 개에 넣고 출력을 `average`로 평균 냄

## 2. 다중 출력 모델

출력층을 여러 개 두고 `outputs`를 리스트로 지정한 함수형 모델 (`keras69_ensemble3`)

예측할 y가 여러 개일 때, 합친 층까지는 공유하고 그 뒤에서 출력마다 층을 나눠(분기) 한 모델로 학습함

1. `Model(inputs=..., outputs=[output1, output2])` : 출력을 리스트로 지정
2. `fit(x, [y1, y2])` : y도 `outputs` 순서대로 리스트로 넣음
3. `compile(loss='mse')` : loss를 하나만 적으면 출력마다 따로 계산하고, 그 합을 줄이도록 학습함

`evaluate`는 loss 합과 출력별 loss를 리스트로 돌려줌

| 순서 | 이름 | 값 |
|---|---|---|
| 0 | `loss` | 출력별 loss의 합 |
| 1 | `(출력1 층 이름)_loss` | 출력 1의 loss |
| 2 | `(출력2 층 이름)_loss` | 출력 2의 loss |

- `return_dict=True`를 주면 위 이름을 키로 한 dict로 돌려줌
- `predict`는 출력마다 배열 하나씩 담은 리스트를 돌려줌

#### cf) 출력에 이어지지 않은 층

`inputs`에서 `outputs`까지 이어진 층만 모델에 들어감. 만들어 놓고 출력까지 잇지 않은 층은 에러 없이 빠짐

- 출력층에 바로 앞 층 대신 더 앞의 층을 넣으면 그 사이 층이 빠짐
- `summary()`에 그 층이 나오지 않고, 파라미터 수도 그만큼 적음